<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 8 — Risk Divergence Analysis (Google Colab)

This notebook helps you complete **Required Task 8** step by step.

## What this task asks you to do
1. Download the **2023 10-K** reports for **Block, Inc.** and **PayPal**
2. Upload them into your LLM workflow
3. Compare their **Risk Factors**
4. Use **multiple prompting techniques**
5. Use an **LLM as a judge**
6. Write a final **200-word executive summary**

Use this notebook from top to bottom, one cell at a time.


## Before you start

### You need:
- A **Google Colab** notebook
- A **Gemini API key** from Google AI Studio
- Two PDF files:
  - Block 2023 10-K
  - PayPal 2023 10-K

### Recommended file names
Rename your downloaded files to:
- `block_2023_10k.pdf`
- `paypal_2023_10k.pdf`

Then upload both files when the upload cell asks you to.


In [1]:
!pip -q install -U google-genai pymupdf pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 45.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which

## Step 1 — Enter your Gemini API key

Replace `YOUR_API_KEY_HERE` with your real Gemini API key.


In [14]:
import os
from google.colab import userdata

# Retrieve the API key securely from Colab secrets
# Please add your Gemini API key to Colab secrets under the name 'GOOGLE_API_KEY'
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

if not os.environ.get("GEMINI_API_KEY"): # Check if the key is actually set and not empty
    print("⚠️ Please set your Gemini API key in Colab secrets under the name 'GOOGLE_API_KEY'.")
else:
    print("✅ API key set from Colab secrets.")

✅ API key set from Colab secrets.


## Step 2 — Upload the two 10-K PDFs

After running the next cell:
1. Click **Choose Files**
2. Upload:
   - `block_2023_10k.pdf`
   - `paypal_2023_10k.pdf`


In [15]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Saving block_2023_10k.pdf to block_2023_10k (1).pdf
Saving paypal_2023_10k.pdf to paypal_2023_10k (1).pdf
Uploaded files: ['block_2023_10k (1).pdf', 'paypal_2023_10k (1).pdf']


## Step 3 — Read the PDFs and extract the Risk Factors section

We will:
- extract text from each PDF
- isolate the text from **Item 1A. Risk Factors**
- stop when the next item starts

This makes the later prompts more focused and reliable.


In [16]:
import fitz
import re
from pathlib import Path

def extract_full_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        pages.append(page.get_text("text"))
    return "\n".join(pages)

def extract_risk_factors_section(full_text):
    text = full_text.replace("\x00", " ")
    text = re.sub(r"\n+", "\n", text)

    start_patterns = [
        r"ITEM\s*1A\.?\s*RISK FACTORS",
        r"Item\s*1A\.?\s*Risk Factors",
    ]

    end_patterns = [
        r"ITEM\s*1B\.?\s*UNRESOLVED STAFF COMMENTS",
        r"ITEM\s*1C\.?\s*CYBERSECURITY",
        r"Item\s*1B\.?\s*Unresolved Staff Comments",
        r"Item\s*1C\.?\s*Cybersecurity",
    ]

    start_match = None
    for pattern in start_patterns:
        m = re.search(pattern, text)
        if m:
            start_match = m
            break

    if not start_match:
        return None

    start_idx = start_match.start()
    after_start = text[start_idx:]

    end_idx_relative = None
    for pattern in end_patterns:
        m = re.search(pattern, after_start[200:])  # skip immediate heading repeats
        if m:
            candidate = 200 + m.start()
            if end_idx_relative is None or candidate < end_idx_relative:
                end_idx_relative = candidate

    if end_idx_relative is None:
        return after_start

    return after_start[:end_idx_relative]

block_pdf = "block_2023_10k.pdf"
paypal_pdf = "paypal_2023_10k.pdf"

block_text = extract_full_text_from_pdf(block_pdf)
paypal_text = extract_full_text_from_pdf(paypal_pdf)

block_risk = extract_risk_factors_section(block_text)
paypal_risk = extract_risk_factors_section(paypal_text)

print("Block risk section extracted:", block_risk is not None)
print("PayPal risk section extracted:", paypal_risk is not None)
print("Block risk section length:", len(block_risk) if block_risk else 0)
print("PayPal risk section length:", len(paypal_risk) if paypal_risk else 0)

Block risk section extracted: True
PayPal risk section extracted: True
Block risk section length: 180982
PayPal risk section length: 89395


## Step 4 — Quick sanity check

Run this to confirm the extracted sections look correct.


In [5]:
print("===== BLOCK RISK FACTORS PREVIEW =====")
print(block_risk[:3000])

print("\n\n===== PAYPAL RISK FACTORS PREVIEW =====")
print(paypal_risk[:3000])

===== BLOCK RISK FACTORS PREVIEW =====
ITEM 1A. RISK FACTORS
Investing in our securities involves a high degree of risk. You should carefully consider the risks and uncertainties described below, together with all of the
other information in this Annual Report on Form 10-K, including the section titled Management’s Discussion and Analysis of Financial Condition and Results of
Operations and our consolidated financial statements and related notes, before making any investment decision with respect to our securities. The risks and uncertainties
described below may not be the only ones we face. If any of the risks actually occur, our business could be materially and adversely affected. In that event, the market
price of our Class A common stock could decline, and you could lose part or all of your investment.
Risk Factors Summary
Our business operations are subject to numerous risks and uncertainties, including those outside of our control, that could cause our actual results to be harmed

## Step 5 — Connect to Gemini

We will use Gemini to analyze the two Risk Factors sections.


In [17]:
from google import genai
from textwrap import shorten

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL_NAME = "gemini-2.5-flash"

def ask_gemini(prompt):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    return response.text

## Step 6 — Create the common analysis context

This is the shared background used in all prompting techniques.


In [18]:
common_context = f'''
You are a financial risk analyst.

Base question:
Is Block's newer AI & Bitcoin / ecosystem-expansion focus creating operational or legal risks that appear more exposed, less mature, or less clearly controlled than PayPal's more conservative approach?

Instructions:
- Compare only the 2023 10-K Risk Factors sections below.
- Focus on subtle wording changes, concentration risk, governance risk, product expansion risk, compliance risk, cyber / data risk, regulatory risk, and execution risk.
- Do not just summarize. Compare.
- Be careful not to invent facts not supported by the text.
- Highlight signals that might point to future legal, operational, or control trouble.
- Say clearly whether the evidence is strong, moderate, or weak.

BLOCK RISK FACTORS:
{block_risk}

PAYPAL RISK FACTORS:
{paypal_risk}
'''

## Step 7 — Prompting technique 1: baseline direct prompt

In [19]:
prompt_1 = common_context + '''

Task:
Write a concise comparative analysis answering the base question.

Output format:
1. Main conclusion
2. 5 strongest risk signals
3. Final verdict in one sentence
'''

response_1 = ask_gemini(prompt_1)
print(response_1)

**1. Main Conclusion**

Based solely on the 2023 10-K Risk Factors, Block's newer AI & Bitcoin / ecosystem-expansion focus presents operational and legal risks that appear more exposed, less mature, and less clearly controlled than PayPal's approach. Block directly engages in activities like Bitcoin custody, operating an industrial bank and broker-dealer, and integrating highly diverse new businesses (payments, lending, music streaming) which inherently subject it to a wider array of nascent regulatory scrutiny, complex compliance obligations, and higher operational execution challenges, often with direct balance sheet exposure. PayPal, while also expanding into crypto and BNPL, often frames its exposure in these areas through partnerships and externalization of certain direct risks, and its risk disclosures lean more towards general industry and technological evolution rather than specific, direct operational incidents or fundamental governance structures observed in Block's filing.



## Step 8 — Prompting technique 2: structured forensic checklist

In [20]:
prompt_2 = common_context + '''

Task:
Perform a forensic-style risk comparison.

Score Block and PayPal from 1 to 5 on each of the following:
- Regulatory exposure
- Operational complexity
- Governance / control maturity
- Product expansion risk
- Legal / compliance ambiguity
- Technology / cyber risk

Then explain:
- Why Block scores higher or lower
- Which exact themes in Block look less stable than PayPal
- Whether the phraseology suggests underdisclosed execution risk

End with:
"Overall answer to the base question: ..."
'''

response_2 = ask_gemini(prompt_2)
print(response_2)

Here's a forensic-style risk comparison of Block and PayPal, focusing on the instructed themes and providing scores based *only* on the provided 2023 10-K Risk Factors sections.

**Risk Comparison Scores (1=Lowest Risk, 5=Highest Risk):**

| Risk Category                 | Block | PayPal |
| :---------------------------- | :---- | :----- |
| Regulatory exposure           | 5     | 4      |
| Operational complexity        | 5     | 4      |
| Governance / control maturity | 5     | 3      |
| Product expansion risk        | 5     | 4      |
| Legal / compliance ambiguity  | 5     | 4      |
| Technology / cyber risk       | 4     | 4      |

---

**Detailed Explanation and Analysis:**

**Why Block scores higher or lower:**

Block generally scores higher (indicating higher risk) than PayPal across most categories. This is primarily attributable to Block's more aggressive and diverse expansion into nascent and highly regulated financial services, as well as distinct non-financial ventures

## Step 9 — Prompting technique 3: red-team / skeptical investigator

In [21]:
prompt_3 = common_context + '''

Task:
Act like a skeptical investigator trying to find hidden future trouble.

Your goal is to identify:
- risks that are easy to overlook
- wording that sounds broad, vague, or defensive
- areas where Block may have greater operational fragility than PayPal
- areas where fast ecosystem expansion could outpace controls

Output format:
- Hidden risk signal 1
- Hidden risk signal 2
- Hidden risk signal 3
- Hidden risk signal 4
- Hidden risk signal 5
- Final answer to the base question
'''

response_3 = ask_gemini(prompt_3)
print(response_3)

Here's an analysis comparing Block's and PayPal's risk factors through a skeptical investigator's lens:

**Hidden risk signal 1: Direct Bitcoin Investment Exposure vs. Customer Facilitation**
*   **Block:** Explicitly states, "Our bitcoin investments being subject to volatile market prices, impairment, and other risks of loss." This directly signals Block's own balance sheet exposure to a highly volatile, indefinite-lived intangible asset that is subject to impairment charges (downward adjustments without upward revisions until sale). The text also mentions "any failure to safeguard the bitcoin we hold on behalf of ourselves and other parties."
*   **PayPal:** Focuses on "Our customer cryptocurrency offerings could subject us to additional regulations, licensing requirements, or other obligations or liabilities" and the PYUSD stablecoin, emphasizing regulatory uncertainty and reliance on a "third-party issuer" and "third-party custodians." PayPal's risks primarily revolve around regula

## Step 10 — Prompting technique 4: comparative table + investment memo style

In [23]:
prompt_4 = common_context + '''

Task:
Write the answer like a short investment-risk memo.

First create a table with these columns:
- Risk Area
- Block
- PayPal
- Which side looks riskier?
- Why it matters

After the table, write a short paragraph answering:
Is Block's AI & Bitcoin / ecosystem strategy producing more concerning operational-risk signals than PayPal's conservative model?

Finish with a confidence level: Low / Medium / High
'''

response_4 = ask_gemini(prompt_4)
print(response_4)

Here's an investment-risk memo comparing Block and PayPal based on their 2023 10-K Risk Factors:

**Investment Risk Memo: Block vs. PayPal - Operational & Legal Risk Comparison (2023 10-K)**

**Date:** October 26, 2023

**Subject:** Comparative Operational and Legal Risk Assessment – Block's AI & Bitcoin/Ecosystem Expansion vs. PayPal's Conservative Approach

This memo analyzes the operational and legal risks highlighted in the 2023 10-K filings of Block and PayPal, with a particular focus on Block's newer AI and Bitcoin/ecosystem-expansion strategy.

| Risk Area            | Block (Signals from 2023 10-K)                                                                                                                                                                                                                                                                                                                                                                                                   

## Step 11 — Collect all four answers together

In [24]:
all_responses = {
    "baseline_direct": response_1,
    "forensic_checklist": response_2,
    "red_team": response_3,
    "investment_memo": response_4,
}

for name, text in all_responses.items():
    print("\n" + "="*80)
    print(name.upper())
    print("="*80)
    print(text[:4000])


BASELINE_DIRECT
**1. Main Conclusion**

Based solely on the 2023 10-K Risk Factors, Block's newer AI & Bitcoin / ecosystem-expansion focus presents operational and legal risks that appear more exposed, less mature, and less clearly controlled than PayPal's approach. Block directly engages in activities like Bitcoin custody, operating an industrial bank and broker-dealer, and integrating highly diverse new businesses (payments, lending, music streaming) which inherently subject it to a wider array of nascent regulatory scrutiny, complex compliance obligations, and higher operational execution challenges, often with direct balance sheet exposure. PayPal, while also expanding into crypto and BNPL, often frames its exposure in these areas through partnerships and externalization of certain direct risks, and its risk disclosures lean more towards general industry and technological evolution rather than specific, direct operational incidents or fundamental governance structures observed in 

## Step 12 — Use Gemini as the judge

This is the LLM-as-judge step required by your task.


In [25]:
judge_prompt = f'''
You are an expert evaluator grading four different LLM answers to the same finance-risk question.

Base question:
Is Block's strategy shift toward AI, Bitcoin, and ecosystem expansion creating undisclosed or underappreciated operational risks compared with PayPal's more conservative approach?

Grade each answer on a scale of 1 to 10 for:
1. Accuracy to source material
2. Depth of risk detection
3. Ability to notice subtle wording differences
4. Comparative reasoning
5. Clarity and usefulness

Then:
- rank the 4 answers from best to worst
- explain the strengths and weaknesses of each
- pick the single best answer
- provide concrete advice for writing a final executive summary

ANSWER 1 — baseline_direct
{response_1}

ANSWER 2 — forensic_checklist
{response_2}

ANSWER 3 — red_team
{response_3}

ANSWER 4 — investment_memo
{response_4}
'''

judge_response = ask_gemini(judge_prompt)
print(judge_response)

Here is the expert evaluation of the four LLM answers:

---

### **Evaluation Criteria Definitions:**

1.  **Accuracy to Source Material:** How well do the claims align with typical 10-K risk factor language and financial concepts? (Evaluated based on the *credibility* and *specificity* of the simulated 10-K excerpts/paraphrases, as no actual 10-K was provided).
2.  **Depth of Risk Detection:** How thoroughly and incisively do the answers identify and articulate the nuances of the risks? Do they go beyond surface-level observations?
3.  **Ability to Notice Subtle Wording Differences:** Do they pick up on slight variations in how risks are framed by each company (e.g., "direct investment" vs. "customer facilitation," "source of financial strength" vs. "rely on unaffiliated institutions")? This is crucial for comparative analysis.
4.  **Comparative Reasoning:** How effectively do the answers compare and contrast Block and PayPal, drawing clear distinctions and explaining *why* one is ris

## Step 13 — Generate the final 200-word executive summary

This uses the judge's feedback to produce the final answer required by the assignment.


In [26]:
final_summary_prompt = f'''
You are writing the final answer for a university assignment.

Use:
- the four model responses
- the judge feedback

Write a polished executive summary of about 200 words answering this base question:

"Is Block's new AI & Bitcoin focus creating undisclosed operational risks compared to PayPal's conservative approach?"

Requirements:
- Keep it around 200 words
- Sound professional and analytical
- Do not mention prompting techniques
- Do not mention that an LLM judged the outputs
- Make the conclusion clear
- Distinguish evidence from inference
- Avoid exaggeration
- Focus on operational, legal, governance, execution, and compliance risks

Judge feedback:
{judge_response}

Best available material:
{response_1}

{response_2}

{response_3}

{response_4}
'''

final_executive_summary = ask_gemini(final_summary_prompt)
print(final_executive_summary)

Block's strategic pivot towards AI, direct Bitcoin involvement, and an aggressively expanding ecosystem demonstrably creates a **higher and less mature operational and legal risk profile** compared to PayPal's more conservative approach. This heightened risk, while extensively disclosed, implies significant *underappreciated execution challenges* given the novelty and breadth of Block's ventures.

Key differentiators include Block’s **direct financial and operational exposure**, encompassing proprietary Bitcoin investments subject to volatile markets and "irreversible" private key loss risks, alongside the direct regulatory burden and "source of financial strength" obligations of its industrial bank. This contrasts sharply with PayPal’s strategy of leveraging third-party partnerships to manage similar emerging risks, externalizing direct balance sheet exposure.

Furthermore, Block's **broad and rapid ecosystem expansion** into disparate ventures like music streaming and services for mi

## Step 14 — Save everything

This will save your outputs into text files you can download.


In [27]:
from pathlib import Path

Path("task8_outputs").mkdir(exist_ok=True)

with open("task8_outputs/block_risk_factors.txt", "w", encoding="utf-8") as f:
    f.write(block_risk or "")

with open("task8_outputs/paypal_risk_factors.txt", "w", encoding="utf-8") as f:
    f.write(paypal_risk or "")

with open("task8_outputs/prompt_response_1.txt", "w", encoding="utf-8") as f:
    f.write(response_1)

with open("task8_outputs/prompt_response_2.txt", "w", encoding="utf-8") as f:
    f.write(response_2)

with open("task8_outputs/prompt_response_3.txt", "w", encoding="utf-8") as f:
    f.write(response_3)

with open("task8_outputs/prompt_response_4.txt", "w", encoding="utf-8") as f:
    f.write(response_4)

with open("task8_outputs/judge_feedback.txt", "w", encoding="utf-8") as f:
    f.write(judge_response)

with open("task8_outputs/final_executive_summary.txt", "w", encoding="utf-8") as f:
    f.write(final_executive_summary)

print("✅ Files saved inside task8_outputs/")

✅ Files saved inside task8_outputs/


## Step 15 — Download your final summary

Run this if you want the final executive summary on your computer.


In [28]:
from google.colab import files
files.download("task8_outputs/final_executive_summary.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional troubleshooting

### If the PDF extraction fails
- Check the filenames carefully
- Make sure the files are real 10-K PDFs
- Re-run the upload cell

### If Gemini gives an API error
- Check your API key
- Make sure billing / quota is available for your account
- Re-run the API key cell

### If the result is too generic
- Use the extracted Risk Factors text only, not the whole PDF
- Re-run the four prompting cells
- Tighten the judge prompt if needed
